## **Nugen Intelligence**
<img src="https://nugen.in/logo.png" alt="Nugen Logo" width="200"/>

Domain-aligned foundational models at industry leading speeds and zero-data retention! To learn more, visit [Nugen](https://docs.nugen.in/introduction)

### **Vision Alignmeent with the Nugen API**
 
This cookbook demonstrates how to create a Vision Alignment Project using the Nugen API. You'll learn how to upload an image dataset, automatically generate benchmark questions, train an aligned vision model, monitor training progress, and finally perform inference using the aligned model.

The notebook explains each step in a simple, sequential manner so that you can easily reproduce the complete workflow.

### **Dataset Preparation**

Before creating a Vision Alignment project, your image dataset must be preprocessed.

- Convert every image to a **Base64-encoded string**. 
  Eample-{"image": "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAKAAAAAoCAIAAAD2TmbPAAAL}
- Create a **`.jsonl`** file containing one JSON object per line.
- Each JSON object should include the Base64-encoded image and the corresponding annotation or metadata required for training.
- Save the file with the `.jsonl` extension.
- Use this `.jsonl` file when uploading the dataset for Vision Alignment.

> **Note:** Vision Alignment accepts image datasets in Base64 format stored in a `.jsonl` file. Raw image files (such as `.jpg` or `.png`) must be converted before starting the alignment process.

### **Workflow**                                                                                  
The cookbook covers the following steps:

1 Upload a vision dataset (.jsonl).
2 Retrieve document details.
3 Generate benchmark questions from the uploaded dataset.
4 Create a Vision Alignment project.
5 Monitor alignment training status.
6 Run inference using the aligned vision model.

### **Dataset Split**

After uploading your dataset, 15% of the uploaded data will automatically be used to generate benchmark questions for evaluating the aligned model. The remaining data is used during the alignment process.

Uploaded Dataset
      - 85% → Alignment Training
      - 15% → Benchmark Generation

**Install the required Python packages**

In [1]:
%pip install -q requests


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**Import Required Libraries**

In [2]:
import requests
import json
import time
import base64

**Set up your Nugen API Key**

To get your Nugen API key, visit the [Nugen Platform](https://platform.nugen.in/)

Replace your-nugen-api-key with your actual API key:

In [ ]:
API_KEY = "your-nugen-api-key"

Your API key is required to authenticate requests to the Nugen API.

In [4]:
headers = {"Authorization": f"Bearer {API_KEY}"}

Here, we define the API base URL and your API key. Replace <--nugen api key--> with your actual key to authenticate your requests to the Nugen API. The MODEL variable specifies the model we will use for generating the routines.

In [5]:
BASE_URL = "https://api.nugen.in"

**Upload the Vision Dataset**

Upload a JSONL dataset

In [6]:
url = f"{BASE_URL}/api/v3/documents/create"

In [ ]:
with open("vision_dataset.jsonl", "rb") as f:
    files = {
        "files": ("vision_dataset.jsonl", f, "application/json")
    }

    data = {
        "categories": "image"
    }

    document_response = requests.post(
        url,
        headers=headers,
        data=data,
        files=files
    )
print(document_response.text)


{"document_ids":["document_01m3bq696z7fek67"]}


**Get status of Document uploaded**

Once the upload is complete, retrieve the document ID

The status of a document upload task, at the address a status lives at.

In [8]:
response_data = json.loads(document_response.text) 

id = response_data["document_ids"][0]


In [9]:
document_status_url = f"{BASE_URL}/api/v3/documents/{id}/status"

In [10]:
document_status_response = requests.get(document_status_url, headers=headers)

print(document_status_response.text)

{"status":"READY","document_id":"document_01m3bq696z7fek67","created_at":"2026-09-25T07:24:02.644626","completed_at":"2026-09-25T07:24:02.962729","updated_at":"2026-09-25T07:24:02.962729","progress":null,"eta_seconds":null}


**Generate Benchmark Questions**

Generate evaluation questions from the uploaded dataset.

Note: Benchmark questions are generated using 15% of the uploaded dataset

In [11]:
response_data = json.loads(document_status_response.text) 

document_id = response_data["document_id"]

In [12]:
generate_benchmark_url = f"{BASE_URL}/api/v3/benchmarks/create"

In [14]:
payload = {
    "document_ids": [document_id],
    "num_questions": 20
}

benchmark_response = requests.post(generate_benchmark_url, json=payload, headers=headers)

print(benchmark_response.text)

{"benchmark_id":"benchmark_01m3bq7c5ez9pre1","benchmark_name":"benchmark_01m3bq7c5ez9pre1","status":"PROCESSING"}


**Get status Benchmark Generation**

Check whether benchmark generation has completed.

In [15]:
response_data = json.loads(benchmark_response.text)

benchmark_id = response_data["benchmark_id"]

In [16]:
benchmark_status_url = f"{BASE_URL}/api/v3/benchmarks/{benchmark_id}/status"

In [17]:
response = requests.get(benchmark_status_url, headers=headers)

print(response.text)

{"benchmark_id":"benchmark_01m3bq7c5ez9pre1","status":"READY","created_at":"2026-09-25T07:24:38.428006","completed_at":"2026-09-25T07:24:39.571934","updated_at":"2026-09-25T07:24:39.571934","progress":null,"eta_seconds":null}


**Create a Vision Alignment Project**

**Example base model:**

qwen2-vl-2b-instruct

In [18]:
alignment_url = f"{BASE_URL}/api/v3/alignment-projects/create"

In [19]:
payload = {
    "alignment_name": "My Vision Alignment ",
    "base_model_id": "qwen2-vl-2b-instruct",
    "document_ids": [document_id],
    "workflow_id": "workflow-abc123",
    "benchmark_id": benchmark_id,
    "description": "This project aims to align the model for better vision alignment."
}

print(payload)
alignment_response = requests.post(alignment_url, json=payload, headers=headers)

print(alignment_response.text)

{'alignment_name': 'My Vision Alignment ', 'base_model_id': 'qwen2-vl-2b-instruct', 'document_ids': ['document_01m3bq696z7fek67'], 'workflow_id': 'workflow-abc123', 'benchmark_id': 'benchmark_01m3bq7c5ez9pre1', 'description': 'This project aims to align the model for better vision alignment.'}
{"alignment_id":"alignment_01m3bq98kv60dxyt","status":"PROCESSING"}


**Check Alignment Status**

Track the alignment status.

In [23]:
response_data = json.loads(alignment_response.text)

alignment_id = response_data["alignment_id"]

In [24]:
alignment_status_url = f"{BASE_URL}/api/v3/alignment-projects/{alignment_id}/status"

In [25]:
while True:
    alignment_status_response = requests.get(alignment_status_url, headers=headers)
    alignment_status_response.raise_for_status()
    print(alignment_status_response.text)
    data = alignment_status_response.json()
    status = data["status"]

    print(f"Current status of Alignment: {status}")

    if status == "READY":
        print("Alignment completed.")
        break

    if status == "FAILED":
        raise Exception("Alignment failed.")
    time.sleep(10)

{"alignment_id":"alignment_01m3bq98kv60dxyt","status":"READY","created_at":"2026-09-25 07:25:40.200388","completed_at":"2026-09-25 07:32:44.610358","updated_at":"2026-09-25 07:32:44.610358","progress":null,"eta_seconds":null,"queue_position":null,"stop_requested_at":null,"early_deployable":false}
Current status of Alignment: READY
Alignment completed.


### **List Aligned Model**

Retrieve all domain-aligned models for the authenticated user.

In [26]:
list_aligned_model_url= f"{BASE_URL}/api/v3/models/aligned"

In [27]:
list_aligned_model_response = requests.get(list_aligned_model_url, headers=headers)

print(list_aligned_model_response.text)

{"domain_aligned_models":[{"model_id":"model_01m3bqd0kpqdsngr","model_name":"model_my_vision_alignment_","base_model_id":"qwen2-vl-2b-instruct","base_model_name":"Qwen2-Vl-2b-Instruct","deployment_status":"UNDEPLOYED","created_at":"2026-09-25 07:31:44.988050","alignment_id":"alignment_01m3bq98kv60dxyt"},{"model_id":"model_01m36p4rvbw3fdvs","model_name":"model_textfile-llama-v3p2-3b-reasoning-aligned","base_model_id":"llama-v3p2-3b-reasoning","base_model_name":"Llama-V3p2-3b-Reasoning","deployment_status":"UNDEPLOYED","created_at":"2026-09-23 08:31:33.548715","alignment_id":"alignment_01m36p2kr7e47mrw"}]}


### **Get Model**

Get one aligned model by ID.

In [29]:
response_data = json.loads(list_aligned_model_response.text)

model_id = response_data["domain_aligned_models"][0]["model_id"]

In [30]:
get_aligned_model_url= f"{BASE_URL}/api/v3/models/{model_id}"

In [31]:
get_aligned_model_response = requests.get(get_aligned_model_url, headers=headers)

print(get_aligned_model_response.text)

{"model_id":"model_01m3bqd0kpqdsngr","model_name":"model_my_vision_alignment_","base_model_id":"qwen2-vl-2b-instruct","base_model_name":"Qwen2-Vl-2b-Instruct","alignment_id":"alignment_01m3bq98kv60dxyt","deployment_status":"UNDEPLOYED","error":null,"progress":null,"eta_seconds":null,"performance_metrics":{"accuracy":0.0,"uncertainty":0.0,"domain_violations":0,"average_response_time":0.0},"evaluation_data":{"evaluation_id":"evaluation_01m3bqmg0218nssd","status":"READY","metrics":{},"raw_answers_count":0,"completed_at":null,"created_at":"2026-09-25T07:31:48.355476","method":"eval-compare","baseline_model_id":"qwen2-vl-2b-instruct","base_model":null,"eval_model":null,"comparison":{"metrics":[{"base":0.0,"metric":"binary_correctness_mean","evaluated":0.0,"improvement_%":0},{"base":0.0,"metric":"answer_relevance_mean","evaluated":0.0,"improvement_%":0}],"summary":{"avg_improvement":0.0,"max_improvement":0,"min_improvement":0}},"all_evaluations":[{"evaluation_id":"evaluation_01m3bqmg0218nssd

**Deploy Aligned Model**

Once alignment completes, you'll be able to deploy the model.


In [32]:
response_data = json.loads(get_aligned_model_response.text)

model_id = response_data["model_id"]

In [33]:
deploy_url = f"{BASE_URL}/api/v3/models/{model_id}/deployment"

In [34]:
deploy_response = requests.post(deploy_url, headers=headers)

print(deploy_response.text)

{"model_id":"model_01m3bqd0kpqdsngr"}


**Deploy Status**

Check the deployment status of an aligned model.

In [35]:
response_data = json.loads(deploy_response.text)

model_id = response_data["model_id"]

In [36]:
deploy_status_url = f"https://api.nugen.in/api/v3/models/{model_id}/deployment/status"

In [37]:
deploy_status_response = requests.get(deploy_status_url, headers=headers)

print(deploy_status_response.text)

{"model_id":"model_01m3bqd0kpqdsngr","status":"DEPLOYED","error":null,"created_at":"2026-09-25T07:34:45.895326","completed_at":"2026-09-25T07:34:45.907875","updated_at":"2026-09-25T07:34:45.907877","progress":null,"eta_seconds":null}


### **Get Model**

Get one aligned model by ID.

In [38]:
response_data = deploy_status_response.json()

model_id = response_data["model_id"]

In [39]:
get_model_url = f"{BASE_URL}/api/v3/models/{model_id}"

In [40]:
get_model_response = requests.get(get_model_url, headers=headers)

print(get_model_response.text)

{"model_id":"model_01m3bqd0kpqdsngr","model_name":"model_my_vision_alignment_","base_model_id":"qwen2-vl-2b-instruct","base_model_name":"Qwen2-Vl-2b-Instruct","alignment_id":"alignment_01m3bq98kv60dxyt","deployment_status":"DEPLOYED","error":null,"progress":null,"eta_seconds":null,"performance_metrics":{"accuracy":0.0,"uncertainty":0.0,"domain_violations":0,"average_response_time":0.0},"evaluation_data":{"evaluation_id":"evaluation_01m3bqmg0218nssd","status":"READY","metrics":{},"raw_answers_count":0,"completed_at":null,"created_at":"2026-09-25T07:31:48.355476","method":"eval-compare","baseline_model_id":"qwen2-vl-2b-instruct","base_model":null,"eval_model":null,"comparison":{"metrics":[{"base":0.0,"metric":"binary_correctness_mean","evaluated":0.0,"improvement_%":0},{"base":0.0,"metric":"answer_relevance_mean","evaluated":0.0,"improvement_%":0}],"summary":{"avg_improvement":0.0,"max_improvement":0,"min_improvement":0}},"all_evaluations":[{"evaluation_id":"evaluation_01m3bqmg0218nssd",

**Run Inference**

Once alignment completes, you'll receive an Aligned Model ID.
Use this model for inference.

In [41]:
inference_url = f"{BASE_URL}/api/v3/inference/chat/completions"

The following image_b64 converted image is provided only as an example. Replace it with your own image_b64 image to run inference.

In [42]:
image_b64 ="iVBORw0KGgoAAAANSUhEUgAAAHgAAAAyCAIAAAAYxYiPAAAHS0lEQVR4nO2YW0gU3x/Azzkzo3s1V/CSa0goPURCUVFSalpZD0pFBUpEFCiRPdQ+Jyj0qBK9JEKQtfnkBknaTSV8CyEIfChzH3owg71MM66z4865/B/Ov21z10ulY7/fbz4Py87MmXPOfM453++cgYwxYLHxoF+9gTG2vmOzcm3pza17B8wBpnd6uceAEK5785RShFYfbMbYRrRuJhkeEi4DAIAxNj8/L8vyuswpbjkUClFK06/yJiKRCKUUQsjLxONxWZbn5+f/cZN6qWhK6efPnxVFicVic3NzwWDwy5cvsiyrqgoAWFhY2LFjx/79+xcWFsBqq35lCCEIoUAg0NbWlkgk0l3zM2/fvj116pSu6xhjxlhvb++uXbsqKipisdgfdsBsWAqU0sXFxa6ursLCQkmSzpw5093dffbs2eLi4m3btmmapihKfn5+MBiklFJK2e+CMaaUvn79GiEUjUYZY4SQZB+SxRKJBGPs+vXrVVVV/C5N02ZmZgoLCyORyJLCfzlLRfM/giDU1tbqus4P+bOpqqqqqtfrVRSFMWYYRuJn1t4qn56HDh3y+/3JQ/7La+Y9IYQQQhRFsdvtw8PD/KqqqiUlJemi6TL8hpSNIEOMjkajhJCDBw9KkhSNRnVdz8/PBwBACPk9fD6Koij9zBrXECFEEISnT59OTU1duHCBMSYIAgCA/0ajUVEUeUpACBFCcnJyzp8/PzAwwG/nI5q+LldILX8DYuoBxliSpL6+PoRQY2MjQqijo2PLli3t7e2PHj1yOp3RaJTnJYTQ8+fPh4eH3W63pmmEEJvNdvv2bZvNxlZ7Q+CiR0dHz507RwhJ5rrx8fGurq6PHz+2tLTcuHFDEAQIYXZ2NgCgqanp0qVLqqrm5ORkrBNC+O3bN15Vchj4f4/Hw/9srnRxyTFjTJZlSun79++HhoYeP348NTWVlZV1/Phx8L3rNpttcHCwpaWlqalpZGSkoqIiNzc3fZal1/z/JkXRMIzJycmamhpBEBKJhN1u7+zs7OzsDAQCVVVV+fn527dv1zRtenq6u7sbAHDkyJHs7OxEIpFeLR+2N2/eXLlyhS+1ZEOCIBBCampqHjx4kFwlm0YyiPAuaprmcrnq6upmZmZ6e3s9Hs/CwgJjzDAMxlgkEvF6vbIsNzc3B4NBxtiePXtWDU/pgVLTNABAe3s7vzo2NiaKYkdHB7/q8/mqq6sbGxs/ffrEz4RCIa/XGw6HGWPhcNjr9SZjNK9cURRFUebn59U0QqFQMtNuIuIS6bqux2KxysrKsrKynJyceDxus9kwxjyAgu/RkEfMoaGhYDAoy7LL5WKZ1iaEkEcAxlgoFEII5eXlIYRYyvSHEI6MjGCMfT4fxhhCWFBQMDExcfTo0dLSUj5hwWrbKLvdzlN3egGHw7GWPdFG80M0D9ADAwOSJJ08eRIA4Ha729raEEKpaiCEGGMAAGMsEAg0Nzd7PB7DMLKysjI2QCmNRCI+n+/Dhw+qqu7du/f+/fuCIBw4cGBxcZGXcTgckiQ5HA7uNBwO8yAOACCEAACcTqckSRk9YoxFUbx7925PT48kSYZhJC/xAXa73ZOTk263O+NUMI0foiGE09PT9+7dMwxj69atsix7PJ4V7tR13e/3+3w+AABCSNf1W7du6bqeWkYQhDt37jx8+HB8fLyqqqq+vv7EiROUUofDsW/fvnA4zPMqj9qzs7Mul+vZs2f9/f0ul6uvr6+8vLyurg4AMDEx8fXr14wvNqIoAgCuXbt2+fLl1EwIvouGELpcLrAxnxDWzg/RlNKXL182NDQ0Nzc/efLk8OHDlZWVyZWbCn8YwzAKCgpqa2sBAMutTYQQxri8vHz37t0lJSU7d+4sLi7mOa2+vr61tZUvo5s3bzLGqqurIYQXL1589+5da2trf3//4OCgYRiSJAUCgfLycqfTudxj2O12u93+5zo2kDXGcp5zIpFISUkJ37BgjHmeXJWrV6/29fXxUPPq1Stem2EYdrt9bGwsWUyWZf4VhaVsFDHGiqLYbLbR0VF+5lc3LH/JnuUn0cnNnmEYSzJ1UnRBQUHqFjy1WCINnqBevHiRl5dXWFh47Nix2dlZQgjGmBDi9/srKio0TcMYJzeWyVc09i/egq8AfyRVVYuKisrKyvj3s7W/NsViMa4mWRW/t66u7vTp03xcU2cfpZTvyIeHhxsaGuLx+OLiIqW0p6enuLi4tLRUVVX2jxKd4Xv0ynEmFothjHNzc9eeW1I/OrPvqZ83jxCam5srKipKr42XjEQiHo8HIcQricfjuq6Louhyuf6e7fVa+DXRvw1v5bfVsM3eQP85vyz6D5VlrHCF2tKbW/cOmINJM9pi8/em/xEs0SZhiTYJS7RJWKJNwhJtEpZok7BEm4Ql2iQs0SZhiTYJS7RJWKJNwhJtEpZok7BEm4Ql2iQs0SZhiTYJS7RJWKJNwhJtEpZok7BEm4Ql2iQs0SZhiTYJS7RJ/A+1xXkeN7nnKAAAAABJRU5ErkJggg=="

In [43]:
payload = {
    "model": model_id,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "tell me about this image"
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_b64}"
                    }
                }
            ]
        }
    ],
    "max_tokens": 100,
    "temperature": 1,
    "stream": True,
}

response = requests.post(
    inference_url,
    headers=headers,
    json=payload
)

for line in response.iter_lines():
    if line:
        print(line.decode("utf-8"))

data: {"id": "chatcmpl-model_01m3bqd0kpqdsngr-1790321723", "object": "chat.completion.chunk", "created": 1790321723, "model": "model_01m3bqd0kpqdsngr", "choices": [{"index": 0, "delta": {"content": ""}, "finish_reason": null}]}
data: {"id": "chatcmpl-model_01m3bqd0kpqdsngr-1790321723", "object": "chat.completion.chunk", "created": 1790321723, "model": "model_01m3bqd0kpqdsngr", "choices": [{"index": 0, "delta": {"content": "F "}, "finish_reason": null}]}
data: {"id": "chatcmpl-model_01m3bqd0kpqdsngr-1790321723", "object": "chat.completion.chunk", "created": 1790321723, "model": "model_01m3bqd0kpqdsngr", "choices": [{"index": 0, "delta": {"content": "[ "}, "finish_reason": null}]}
data: {"id": "chatcmpl-model_01m3bqd0kpqdsngr-1790321723", "object": "chat.completion.chunk", "created": 1790321723, "model": "model_01m3bqd0kpqdsngr", "choices": [{"index": 0, "delta": {"content": "r "}, "finish_reason": null}]}
data: {"id": "chatcmpl-model_01m3bqd0kpqdsngr-1790321723", "object": "chat.complet

**Explanation**

Vision Alignment enables you to create a Vision Alignment Model using your own image dataset, allowing the model to better understand and respond to domain-specific visual content. Instead of relying solely on the model's general knowledge, alignment adapts the model to your organization's data and use case.

The Vision Alignment workflow in this cookbook consists of the following steps:

1. Upload a vision dataset in JSONL format.
2. Generate benchmark questions using **15% of the uploaded dataset** to evaluate the aligned model.
3. Use the remaining **85% of the dataset** for alignment training.
4. Create an alignment project by selecting an alignment-ready Vision Language Model.
5. Monitor the training progress until the alignment is complete.
6. Perform inference using the newly aligned model.

By following this workflow, you can create a vision model that produces more accurate and context-aware responses for your specific application.

**Conclusion**

In this cookbook, you learned how to build a complete Vision Alignment pipeline using the Nugen API. Starting with a vision dataset, you uploaded the data, generated benchmark questions-answers, created an alignment project, monitored the alignment process, and finally used the aligned model for inference.

This workflow provides a simple and effective way to adapt a Vision Language Model to domain-specific image understanding tasks.